# Si-Ge cluster expansion workflow - part 2

This is a CASM project tutorial to generate a phase diagram using a Si-Ge binary alloy cluster expansion fit to DFT calculations. The overall workflow is split into two parts.

Topics covered in part 2:

1. **Basis set construction**: Specify clusters and basis functions and construct a Clexulator
2. **Cluster expansion fitting**: Collect energies from import and mapping results and evaluate correlations, the per unitcell mean value of the symmetrically equivalent cluster functions. Fit coefficients to DFT calculated energies
4. **Monte Carlo simulations**: Run semi-grand canonical Monte Carlo simulations using the cluster expansion

<div style="background-color: #ffebb9; padding: 10px; margin: 10px;">
    <b style="color: #B27000;">Important:</b> 
    <p>Using the CASM Clexulator requires a compiler with support for C++17. For GCC, version 10 or later is required.</p>
    <p>If the compiler is made available using a command like <em>module load GCC/13.1.0</em> make sure to perform that step before launching this notebook.</p>
</div>


## Setup

### Environment configuration

When the CASM Clexulator is compiled and linked it must be able to find where CASM is installed. In most cases, the following should be all that is needed. For additional configuration help, see [Environment variable configuration](https://prisms-center.github.io/CASMcode_pydocs/casm/bset/2.0/installation.html#environment-variable-configuration).

In [1]:
# Configuration so that the CASM Clexulator can be compiled 

# Typically, this will work:
import os
res = !python -m libcasm.casmglobal --prefix
os.environ["CASM_PREFIX"]=res[0]
print(os.environ["CASM_PREFIX"])

# In some cases additional flags must be set.
# If the "Generate and compile the Clexulator" step in this notebook fails,
# see environment variable configuration help in link above.

/Users/bpuchala/.local/conda/envs/CASM_v2_py313/lib/python3.13/site-packages/libcasm


### Imports and paths

In [2]:
import pathlib
import numpy as np
import libcasm.xtal as xtal
import libcasm.configuration as casmconfig
from libcasm.xtal import pretty_json
from casm.project import Project
from casm.tools.shared.json_io import read_required, safe_dump

input_dir = pathlib.Path("input")
project_path = pathlib.Path("SiGe_occ")

# Check environment variables:
import os
print(f"CASM_PREFIX={os.environ['CASM_PREFIX']}")

CASM_PREFIX=/Users/bpuchala/.local/conda/envs/CASM_v2_py313/lib/python3.13/site-packages/libcasm


### Setup checks

- This notebook depends the import results obtained in part 1. Here we check that the necessary results exist. 

In [3]:
# Construct project
project = Project.init(path=project_path)

# Enumeration and calculation types
enum_id = "occ_by_supercell.1"
calctype_id = "vasp-parameter-set-1"

enum = project.enum.get(enum_id)

# Calculations dir
target_dir = enum.calctype_dir(calctype_id)

# Imported structures
import_results_path = pathlib.Path(str(target_dir) + ".results.json")
import_results = read_required(import_results_path)

# Mapped configurations
mapped_structures_path = enum.enum_dir / f"mapping_results.{calctype_id}.json"
mapping_results = read_required(mapped_structures_path)

# Fitting data (formation energy, composition)
fitting_data_path = enum.enum_dir / f"fitting_data.{calctype_id}.json"
fitting_data = read_required(fitting_data_path)
formation_energy_per_unitcell = np.array(fitting_data.get("formation_energy_per_unitcell"))
comp_a = np.array(fitting_data.get("comp_a"))
relpath_list = fitting_data.get("relpath")

if len(mapping_results) != 214:
    raise ValueError(
        f"Expected 119 mapped structures, found {len(mapping_results)}. "
        "Try removing the SiGe_occ directory and re-running part 1."
    )


print(f"Found {len(mapping_results)} mapped structures")

CASM project already exists at SiGe_occ
Using existing project
Found 214 mapped structures


## Basis set construction

### Basis set generating group

The formation energy is invariant under transformation by prim factor group operations, so we call it the "generating group" for our cluster expansion basis set. 

- [Project.sym]() gives quick access to symmetry information for the project.
- [Project.sym.print_factor_group]() gives a summary of the prim factor group operations.


In [4]:
project.sym.print_factor_group()

0: 1
1: 4⁺ (-0.2500000  0.2500000  0.2500000) 0.25+x, 0.25-x, -0.25-x
2: 4⁺ ( 0.2500000 -0.2500000  0.2500000) -0.25+x, 0.25-x, 0.25+x
3: 4⁺ ( 0.2500000  0.2500000 -0.2500000) 0.25+x, -0.25+x, 0.25-x
4: 4⁻ (-0.2500000  0.2500000  0.2500000) 0.25+x, -0.25-x, 0.25-x
5: 4⁻ ( 0.2500000 -0.2500000  0.2500000) 0.25+x, 0.25-x, -0.25+x
6: 4⁻ ( 0.2500000  0.2500000 -0.2500000) -0.25+x, 0.25+x, 0.25-x
7: 3⁺ 3*x, -x, -x
8: 3⁺ x, x, -3*x
9: 3⁺ x, -3*x, x
10: 3⁺ x, x, x
11: 3⁻ 3*x, -x, -x
12: 3⁻ x, x, -3*x
13: 3⁻ x, -3*x, x
14: 3⁻ x, x, x
15: 2 0.125+x, 0.125, 0.125-x
16: 2 0.125+x, 0.125-x, 0.125
17: 2 0.125, 0.125+y, 0.125-y
18: 2 (-0.0000000  0.0000000  0.5000000) 0.125, 0.125, z
19: 2 (0.0000000 0.5000000 0.0000000) 0.125, y, 0.125
20: 2 ( 0.5000000 -0.0000000  0.0000000) x, 0.125, 0.125
21: 2 x, -x, -x
22: 2 x, -x, x
23: 2 x, x, -x
24: m x, y, x
25: m x, x, z
26: m x, y, y
27: m 2*x, 2*y, -x-y
28: m 2*x, -x+y, -2*y
29: m x, y, -2*x-y
30: g ( 0.5000000 -0.0000000  0.0000000) -0.125+x, 0.125+y, 

### Specify clusters and basis functions

Use [*bset.make_bspecs*]() to construct a cluster expansion basis set for the Si-Ge formation energy. The parameters that may be useful are:

- *max_length*: The maximum site-to-site distance to allow in clusters, by number of sites in the cluster.
    - Example: ``max_length=[0.0, 0.0, 5.0, 4.0]`` specifies that pair clusters up to distance 5.0 and triplet clusters up to distance 4.0 should be included. The null cluster and point cluster values (elements 0 and 1) are arbitrary.
- *custom_generators*: Optionally, specify particular clusters to include regardless of *max_length*
- *occ_site_basis_functions_specs*: Select the occupation site basis functions. The most common options are:
  - "chebychev": An expansion (with correlation values all equal to 0) about the idealized random alloy where the probability of any of the allowed occupants on a particular site is the same.
  - "occupation": An expansion (with correlation values all equal to 0) about the default configuration where each site is occupied by the first allowed occupant in the Prim.occ_dof list.
  - See details [here](https://prisms-center.github.io/CASMcode_pydocs/casm/bset/2.0/usage/basis_function_specs.html#occupation-site-basis-functions)

In [5]:
# Specify the basis set ID
# - Must be alphanumeric and underscores only
bset_id = "default"

# Specify maximum cluster site-to-site distance,
# by number of sites in the cluster
pair_max_length = 10.01
triplet_max_length = 7.27
quad_max_length = 4.0

# Use chebychev site basis functions (+x, -x)
occ_site_basis_functions_specs = "chebychev"

bset = project.bset.get(bset_id)
bset.make_bspecs(
    max_length=[
        0.0,  # null cluster, arbitrary
        0.0,  # point cluster, arbitrary
        pair_max_length,
        triplet_max_length,
        quad_max_length,
    ],
    occ_site_basis_functions_specs="chebychev",
)
bset.commit()

write: SiGe_occ/basis_sets/bset.default/bspecs.json
write: SiGe_occ/basis_sets/bset.default/writer_params.json



### The bspecs.json file

The previous steps created a "bspecs.json" file storing the basis set specifications:

- *cluster_specs*: specifications for which clusters to construct 
  cluster functions on 
- *basis_functions_specs*: specifications for which type of basis
  functions to generate
- *version*: specifies the Clexulator version to write (CASM v2+)

The "bspecs.json" file can also be edited manually, using the format described [here](https://prisms-center.github.io/CASMcode_docs/formats/casm/clex/ClexBasisSpecs/).


In [6]:
bspecs_path = project.dir.bspecs(bset="default")
print(pretty_json(read_required(bspecs_path)))

{
  "basis_function_specs": {
    "dof_specs": {
      "occ": {
        "site_basis_functions": "chebychev"
      }
    }
  },
  "cluster_specs": {
    "generating_group": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47],
    "orbit_branch_specs": {
      "0": {
        "max_length": 0.0
      },
      "1": {
        "max_length": 0.0
      },
      "2": {
        "max_length": 10.01
      },
      "3": {
        "max_length": 7.27
      },
      "4": {
        "max_length": 4.0
      }
    },
    "site_filter_method": "dof_sites"
  }
}



### Generate and compile the Clexulator

The *bset.update* command generates and compiles a Clexulator (cluster expansion calculator).

Options include:

- *no_compile*: generate the basis set so that you can inspect the clusters and functions without compiling the Clexulator
- *only_compile*: re-compile a Clexulator using the existing files

In [7]:
bset.update(
    # no_compile=False,
    # only_compile=False
)

Cleaning generated files:
- No generated files to remove

Generating clexulator...
Building cluster functions...
Building cluster functions DONE
- n_orbits: 44
- n_functions: 44
build time: 1.6187 (s)

Generating variables...
Generating neighborhoods...
Generating neighborhoods DONE
time: 0.2532 (s)

Generating orbit function formulas...
Generating orbit function formulas DONE
time: 0.0178 (s)

Generating site function formulas...
Generating site function formulas DONE
time: 0.9393 (s)

Generating variables DONE
generation time: 1.2114 (s)

Generated files:
- SiGe_occ/basis_sets/bset.default/cluster_functions.json.gz
- SiGe_occ/basis_sets/bset.default/SiGe_occ_Clexulator_default.cc
- SiGe_occ/basis_sets/bset.default/basis.json
- SiGe_occ/basis_sets/bset.default/variables.json.gz
- SiGe_occ/basis_sets/bset.default/generated_files.json

Generating clexulator DONE
generation time: 2.9001 (s)

Compiling clexulator...
-- Compiling: /Users/bpuchala/codes/CASM_v2_source/CASMcode_modules/CASMc

### Inspect the cluster expansion

Print the cluster orbits:

- An "orbit" is the set of symmetrically equivalent objects. The "prototype" is one element in the orbit.
- In the context of periodic cluster expansion, the "multiplicity" of the orbit is the number of equivalent per unit cell (avoiding double counting clusters which include sites in multiple unit cell).
- The "cluster invariant group" is the set of prim factor group operations plus some lattice translation which leave the cluster unchanged.
  - This is the symmetry used to construct cluster functions. 

In [8]:
bset.print_orbits(
    linear_orbit_indices=None,  # use i.e. set(range(1,5)) to print a subset
)

Orbit 0:
- linear_orbit_index: 0
- multiplicity: 1
- sites: (integral coordinates)
  - {[b, i, j, k]}
  - None
- site-to-site distances:
  - None

Orbit 1:
- linear_orbit_index: 1
- multiplicity: 2
- sites: (integral coordinates)
  - {[b, i, j, k]}
  - [0, 0, 0, 0]
- site-to-site distances:
  - None

Orbit 2:
- linear_orbit_index: 2
- multiplicity: 4
- sites: (integral coordinates)
  - {[b, i, j, k]}
  - [0, 0, 0, 0]
  - [1, 0, 0, 0]
- site-to-site distances:
  - 2.424871

Orbit 3:
- linear_orbit_index: 3
- multiplicity: 12
- sites: (integral coordinates)
  - {[b, i, j, k]}
  - [0, 0, 0, 0]
  - [0, 0, 0, 1]
- site-to-site distances:
  - 3.959798

Orbit 4:
- linear_orbit_index: 4
- multiplicity: 12
- sites: (integral coordinates)
  - {[b, i, j, k]}
  - [0, 0, 0, 0]
  - [1, 0, 1, -1]
- site-to-site distances:
  - 4.643275

Orbit 5:
- linear_orbit_index: 5
- multiplicity: 6
- sites: (integral coordinates)
  - {[b, i, j, k]}
  - [0, 0, 0, 0]
  - [0, 1, -1, -1]
- site-to-site distances:
  -

Print the cluster function prototypes:

- These are the cluster functions on the prototype cluster
- For a binary alloy like Si-Ge there is one function per cluster
  - For example, if the default occupation is Si, then the cluster expansion includes terms for Ge (point), Ge-Ge (pair), Ge-Ge-Ge (triplet), etc., but no Si-Ge (pair) term, because that is the same a Ge (point) term
- In general there may be >1 cluster per function, to account for interactions between different combinations of occupations 

In [9]:
bset.display_occ_site_functions()

function_type = "orbit"

bset.display_functions(
    linear_function_indices=list(range(0, 5)),
    max_terms_per_line=4,
    function_type=function_type,
)

Occupation site functions, Occupation site function:
- sublattice: 0, occ_dof: [Si, Ge]
  - $\phi = [-1.00000000, 1.00000000]$

- sublattice: 1, occ_dof: [Si, Ge]
  - $\phi = [-1.00000000, 1.00000000]$



<IPython.core.display.Latex object>

## Cluster expansion fitting

### Calculate correlations

#### Correlations for configurations in a ConfigurationSet

In [10]:
bset_id = "default"
bset = project.bset.get(bset_id)

corr_calculator = bset.make_corr_calculator()
corr = corr_calculator.per_unitcell(enum.configuration_set)
print("corr:")
print(corr)
print("shape:", corr.shape)

corr:
[[ 1.   -1.    1.   ...  1.    1.    1.  ]
 [ 1.    0.   -1.   ... -1.    1.    1.  ]
 [ 1.    1.    1.   ...  1.    1.    1.  ]
 ...
 [ 1.    0.5   0.25 ... -0.25 -1.   -1.  ]
 [ 1.    0.25 -0.5  ...  0.    0.    0.  ]
 [ 1.    0.75  0.5  ...  0.    0.    0.  ]]
shape: (214, 44)


#### Calculate correlations for mapped configurations

In [11]:
bset_id = "default"
bset = project.bset.get(bset_id)

supercell_set = enum.supercell_set

mapped_configs=[]
for relpath, mapping_data in mapping_results.items():
    mapped_config_with_properties = casmconfig.ConfigurationWithProperties.from_dict(
        data=mapping_data.get("mapped_configuration_with_properties"),
        supercells=supercell_set,
    )
    mapped_config = mapped_config_with_properties.configuration
    mapped_configs.append(mapped_config)

corr = corr_calculator.per_unitcell(mapped_configs)
print("corr:")
print(corr)
print("shape:", corr.shape)


corr:
[[ 1.   -1.    1.   ...  1.    1.    1.  ]
 [ 1.    0.   -1.   ... -1.    1.    1.  ]
 [ 1.    1.    1.   ...  1.    1.    1.  ]
 ...
 [ 1.    0.25 -0.5  ...  0.    0.    0.  ]
 [ 1.    0.    0.   ...  0.    1.    1.  ]
 [ 1.    0.25  0.   ...  0.    0.    0.  ]]
shape: (214, 44)


### Fit coefficients

#### Overview

Here we'll:

- calculate correlations for the mapped configurations obtained in part 1,
- use recursive feature elimination and linear regression methods from *scikit-learn* to fit cluster expansion coefficients using 5, 15, or 25 basis functions,
- plot the expansion coefficients,
- use the cluster expansion to predict the formation energies, and
- plot a comparison of the cluster expansion predicted and DFT calculated energies.

### Plotting functions

#### Plot cluster expansion coefficients

In [12]:
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource
from bokeh.transform import factor_cmap
from bokeh.palettes import Category10

def plot_coeff(index, value, n):
    # Create a ColumnDataSource from the data
    source = ColumnDataSource(data=dict(x=index, y=value))
    
    # Create a new plot with a title and axis labels
    p = figure(height=350, title=f"{n} Coefficients",
               x_axis_label="Basis function index", y_axis_label="Coeff. Value",
               toolbar_location=None, tools="hover", tooltips="@x: @y")
    
    # Add a VBar renderer with a custom color map
    # factor_cmap maps colors to categories
    p.vbar(x='x', top='y', width=0.9, source=source,
           line_color='white')
    
    # # Customize the plot appearance
    p.x_range.start = 0
    p.x_range.end = 45
    
    # Show the results
    output_notebook()
    show(p)

#### Plot DFT-calculated vs Clex-predicted energies

In [13]:
def predict_ef(sparse_coeff, config):
    corr = corr_calculator.per_unitcell(config)
    return sparse_coeff * corr

def plot_energy(sparse_coeff):
    """For a choice of coefficients, plot the DFT-calculated and 
    Clex-predicted energies vs parameteric composition (a)

    Parameters
    ----------
    sparse_coeff: libcasm.clexulator.SparseCoefficients
        A SparseCoefficients object, contains the indices of the non-zero 
        coefficients and their values.
    
    """

    clex_formation_energy_per_unitcell = np.array([
        predict_ef(sparse_coeff, config) 
        for config in mapped_configs
    ])
    error = formation_energy_per_unitcell - clex_formation_energy_per_unitcell
    
    # Create a ColumnDataSource from the data
    source = ColumnDataSource(
        data=dict(
            relpath=[str(x) for x in relpath_list],
            comp_a=comp_a, 
            dft_Ef=formation_energy_per_unitcell,
            clex_Ef=clex_formation_energy_per_unitcell,
            error=error,
        )
    )

    tooltips = [
        ("relpath", "@relpath"),
        ("dft_Ef", "@dft_Ef"),
        ("clex_Ef", "@clex_Ef"),
        ("error", "@error"),
        ("comp_a", "@comp_a"),
    ]
    
    # Create a new plot with a title and axis labels
    p = figure(height=350, title=f"Formation energy",
               x_axis_label="Parametric composition (a)", 
               y_axis_label="Formation energy / unitcell",
               tooltips=tooltips)
    
    p.scatter(
        "comp_a", "dft_Ef", source=source,
        marker="circle", fill_color=None, line_color="blue", size=10, alpha=0.5,
        legend_label="DFT",
    )
    p.scatter(
        "comp_a", "clex_Ef", source=source,
        color="red", size=6, alpha=0.5,
        legend_label="Predicted",
    )
    
    # Show the results
    output_notebook()
    show(p)

    # Create a new plot with the error (dft_Ef - clex_Ef)
    p2 = figure(height=350, title=f"Formation energy error",
               x_axis_label="Parametric composition (a)", 
               y_axis_label="(DFT - CLEX) eV / unitcell",
               tooltips=tooltips)
    
    p2.scatter(
        "comp_a", "error", source=source,
        marker="circle", fill_color="blue", line_color="blue", size=6, alpha=0.5,
    )
    
    # Show the results
    output_notebook()
    show(p2)


### Exercise: Using a varying number of terms

In [14]:
import libcasm.clexulator as casmclex
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline

def fit_n_coeff(n):
    """Fit n non-zero coefficients using LinearRegression w/ RFE and plot results

    Parameters
    ----------
    n: int
        Number of non-zero coefficients to fit.

    Returns
    -------
    sparse_coeff: libcasm.clexulator.SparseCoefficients
        A SparseCoefficients object, contains the indices of the non-zero 
        coefficients and their values.
    
    """
    print("#################################################")
    print(f"Fit using {n} non-zero coefficients...")
    print()
    
    X = corr
    y = formation_energy_per_unitcell
    
    # Construct LinearRegression estimator
    estimator = LinearRegression()
    
    # Construct RFE (Recursive feature elimination) feature selector
    # We want to select n features, and remove 1 feature at each step.
    n_features_to_select = n
    rfe_selector = RFE(
        estimator=estimator, 
        n_features_to_select=n_features_to_select, 
        step=1,
    )
    
    rfe_selector.fit(X, y)
    final_estimator = rfe_selector.estimator_
    
    # Get selected coefficients
    value = final_estimator.coef_
    index = np.where(rfe_selector.support_)[0]

    # Get constant term
    if hasattr(final_estimator, 'intercept_'):
        index = np.array([0] + index.tolist(), dtype=int)
        value = np.array([final_estimator.intercept_] + value.tolist())

    # Perform cross-validation:
    
    # Create a pipeline...
    pipeline = Pipeline([
        ('feature_selection', rfe_selector),
        ('regressor', LinearRegression())
    ])

    # KFold, with `n_splits` random splits into training and test data
    n_splits = 10
    kf = KFold(n_splits=n_splits, shuffle=True)
    
    # Perform cross-validation to get the scores
    print(f"\nPerforming {n_splits}-Fold Cross-Validation with random splits...")
    scoring = make_scorer(mean_squared_error, greater_is_better=True)
    scores = cross_val_score(pipeline, X, y, cv=kf, scoring=scoring)

    cv_score = np.sqrt(np.mean(scores))
    print(f"CV score: {cv_score:.3f} (eV / unit cell)")

    
    # Print selected coefficients: index, value
    # for x in zip(index, value):
    #     print(x[0], x[1])
    
    plot_coeff(index, value, n)

    return casmclex.SparseCoefficients(
        index=index,
        value=value,
    )

sparse_coeff = fit_n_coeff(5)
plot_energy(sparse_coeff)

sparse_coeff = fit_n_coeff(15)
plot_energy(sparse_coeff)

sparse_coeff = fit_n_coeff(25)
plot_energy(sparse_coeff)

sparse_coeff = fit_n_coeff(40)
plot_energy(sparse_coeff)


#################################################
Fit using 5 non-zero coefficients...


Performing 10-Fold Cross-Validation with random splits...
CV score: 0.003 (eV / unit cell)


Loading BokehJS ...

Loading BokehJS ...

Loading BokehJS ...

#################################################
Fit using 15 non-zero coefficients...


Performing 10-Fold Cross-Validation with random splits...
CV score: 0.001 (eV / unit cell)


...

Loading BokehJS ...

#################################################
Fit using 40 non-zero coefficients...


Performing 10-Fold Cross-Validation with random splits...
CV score: 0.001 (eV / unit cell)


Loading BokehJS ...

Loading BokehJS ...

Loading BokehJS ...

## Monte Carlo simulations

### Overview

**Semi-grand canonical Monte Carlo input:**

- System parameters:
  - Prim
  - Cluster expansion
  - Choice of composition axes
- State parameters:
  - Configuration: 
    - Supercell size and shape
    - Initial configuration
  - Thermodynamic conditions:
    - Chemical potential, $\mu$
    - Temperature, $T$
- Sampling parameters:
  - Quantities to sample
  - How often to sample
  - Convergence criteria (requested precision)
  - Run limits (max runtime, max number of samples, etc.)
  - Output options
  
**Semi-grand canonical Monte Carlo output:**

- Ensemble averages:
  - Mean composition, $\langle x \rangle$
  - Mean formation energy, $\langle E_f \rangle$
  - Mean semi-grand canonical energy, $\langle \Omega \rangle$
- Fluctuations:
  - Heat capacity
  - Susceptibility 
- Convergence check results:
  - Number of samples
  - Estimated precision
- Optional output:
  - Snapshots of the configuration when samples are taken
  - All individual sample values

**Output files**

Written to sampling fixture's output directory:

- *status.json*: Periodically written with current completion check results
- *summary.json*: Main output file with statistics and convergence check results. Each subsequent run will append results to this file. Also includes conditions (i.e. chemical potential and temperature) for each run for use in plotting and analysis.

### Load a Monte Carlo System

Specify the system parameters:

In [15]:
from casm.tools.shared.json_io import read_required
from libcasm.clexmonte import (
    MonteCalculator,
    System,
    make_initial_state,
)

system_data = read_required(input_dir / "system.json")
bset_dir = project.dir.bset_dir(bset=bset_id)

system = System.from_dict(
    data=system_data,
    search_path=[str(input_dir), str(bset_dir)],
)

# construct a semi-grand canonical MonteCalculator
calculator = MonteCalculator(
    method="semigrand_canonical",
    system=system,
)

### Exercise: Run at T=100, 200, 300, 600 K

At each temperature, run a series of Monte Carlo calculations at varying chemical potential.

**Reminder:** Results are appended to the *summary.json* files. To re-run, delete the output directories *mc/output.thermo.\<temp\>/* first.

In [16]:
import numpy as np
import sys

# Run at several chemical potentials, w/ dependent runs:
mu_list = np.arange(0.05, -0.051, step=-0.01)
temp_list = [100, 200, 300, 600]
line_colors = ["blue", "cyan", "red", "green"]

for temp in temp_list:

    # construct the initial state (default configuration)
    initial_state, motif, motif_id = make_initial_state(
        calculator=calculator,
        conditions={
            "temperature": 300.0,      # default value
            "param_chem_pot": [-1.0],  # default value
        },
        min_volume=1000,
    )
    
    state = initial_state

    # construct default sampling fixture parameters
    label="thermo"
    output_dir = project.path / "mc" / f"output.{label}.{temp}"
    thermo = calculator.make_default_sampling_fixture_params(
        label=label,
        output_dir=str(output_dir),
    )

    # set convergence level for potential_energy
    thermo.converge(quantity="potential_energy", abs=1e-3)

    # set convergence level for param_composition("a")
    thermo.converge(quantity="param_composition", abs=1e-3, component_name=["a"])

    print(xtal.pretty_json(thermo.to_dict()))
    sys.stdout.flush()
    
    for mu in mu_list:
        state.conditions.scalar_values["temperature"] = temp
        state.conditions.vector_values["param_chem_pot"] = [mu]
        sampling_fixture = calculator.run_fixture(
            state=state,
            sampling_fixture_params=thermo,
        )
        print(f"T: {temp}, mu:{mu}")
        sys.stdout.flush()

print("DONE")


{
  "analysis": {
    "functions": ["heat_capacity", "mol_susc", "param_susc", "mol_thermochem_susc", "param_thermochem_susc"]
  },
  "completion_check": {
    "begin": 100,
    "convergence": [
      {
        "abs_precision": 0.001,
        "component_index": [0],
        "component_name": ["a"],
        "quantity": "param_composition"
      },
      {
        "abs_precision": 0.001,
        "component_index": [0],
        "component_name": ["0"],
        "quantity": "potential_energy"
      }
    ],
    "cutoff": {},
    "period": 100,
    "spacing": "linear"
  },
  "log": {
    "file": "/Users/bpuchala/codes/CASM_v2_source/CASMcode_modules/CASMcode_project/notebooks/SiGe_occ/mc/output.thermo.100/status.json",
    "frequency_in_s": 600.0
  },
  "results_io": {
    "kwargs": {
      "output_dir": "/Users/bpuchala/codes/CASM_v2_source/CASMcode_modules/CASMcode_project/notebooks/SiGe_occ/mc/output.thermo.100",
      "write_observations": false,
      "write_trajectory": false
    },
  

In [17]:
from casm.tools.shared.json_io import read_required

def plot_comp(temp_list, line_colors):

    # Create a new plot with a title and axis labels
    p = figure(height=350, title=f"Monte Carlo results",
               x_axis_label="Parametric chemical potential", 
               y_axis_label="Parameteric composition")

    # Create a new plot with a title and axis labels
    p2 = figure(height=350, title=f"Monte Carlo results",
               x_axis_label="Parametric chemical potential", 
               y_axis_label="Mol composition")

    index = 0
    for temp in temp_list:
        label="thermo"
        output_dir = project.path / "mc" / f"output.{label}.{temp}"
        summary_path = output_dir / "summary.json"
        summary_data = read_required(summary_path)

        comp_a = np.array(summary_data["statistics"]["param_composition"]["a"]["mean"])
        comp_Si = np.array(summary_data["statistics"]["mol_composition"]["Si"]["mean"])
        comp_Ge = np.array(summary_data["statistics"]["mol_composition"]["Ge"]["mean"])
        mu = np.array(summary_data["conditions"]["param_chem_pot"]["a"])
    
        # Create a ColumnDataSource from the data
        source = ColumnDataSource(
            data=dict(
                comp_a=comp_a, 
                comp_Si=comp_Si,
                comp_Ge=comp_Ge,
                mu=mu,
            )
        )
        
        p.line(
            "mu", "comp_a", source=source,
            line_color=line_colors[index],
            line_width=2,
            legend_label=f"{temp} K",
        )

        p2.line(
            "mu", "comp_Si", source=source,
            line_color=line_colors[index],
            line_width=2,
            line_dash="solid",
            legend_label=f"Si-{temp} K",
        )
        p2.line(
            "mu", "comp_Ge", source=source,
            line_color=line_colors[index],
            line_width=2,
            line_dash="dashed",
            legend_label=f"Ge-{temp} K",
        )
        index += 1
    
    # Show the results
    output_notebook()
    show(p)
    show(p2)

plot_comp(temp_list, line_colors)     
    
    

Loading BokehJS ...